# 02_gold
Goal: build the business tables that answer the main question:
**where do we lose our customers, before they buy or after their first order?**

- **Input:** `dev.ecommerce.events_silver` · 233,080,143 rows · one row = one event (view / cart / purchase)
- **Language:** SQL (`CREATE OR REPLACE TABLE ... AS SELECT`). Safe to run again: tables are rebuilt, never appended.
- **Findings are not here.** This notebook only builds and checks tables. Findings are in `03_analysis`.

## Tables
| Table | One row is | Rows | Status |
|---|---|---|---|
| `gold_orders` | one order | 3,063,118 | ✅ done |
| `gold_funnel` | one month (Dec 2019, Jan 2020) | 2 | ✅ done |
| `gold_funnel_category` | one month + one main category | 28 | ✅ done · ⚠️ labels not reliable |
| `gold_funnel_brand` | one month + one brand | | ✅ done |
| funnel by price | one month + one price band | | ⬜ next |
| cart abandonment (per session) | | | ⬜ |
| overview | | | ⬜ |
| retention | | | ⬜ |

## gold_orders
An order = one user's purchases in one session.
`order_id` = user_session + '_' + user_id, because one session had 2 users.
- Columns: `order_id`, `user_id`, `order_time` (first purchase time), `item_count` (one purchase row = one item), `order_value`
- 3,063,118 orders · `order_id` is unique · 0 purchases with a NULL price
- 1,203 orders have 10+ items (0.04% of orders, 0.58% of revenue). Maybe resellers. Kept, because the effect is small.

## gold_funnel
The funnel counts **users**, not sessions or visits: a user is counted once per month (`count(DISTINCT user_id)`).
- **December and January only.** Before 8 Nov, cart tracking was not reliable: in October more people bought (347,118)
  than added to cart (337,117), and on 8 Nov carts jumped from 18,908 to 74,953 in one day while purchases stayed the same.

| Column | Meaning |
|---|---|
| `viewers` / `carters` / `buyers` | users who viewed / added to cart / bought in that month |
| `lost_at_cart` | viewers - carters |
| `lost_at_purchase` | carters - buyers |
| `cart_rate_pct` | carters / viewers · one step: view → cart |
| `cart_to_purchase_pct` | buyers / carters · one step: cart → purchase |
| `purchase_rate_pct` | buyers / viewers · the whole way: view → purchase |

- **Checks:** one row per month · viewers ≥ carters ≥ buyers · lost columns add up · all rates between 0 and 100
- **Limits:** about 1% of buyers have no cart event · the data ends on 31 Jan

## gold_funnel_category
The same funnel as `gold_funnel`, split by **main category** (the part of `category_code` before the first dot, `split_part(category_code, '.', 1)`).
- **One row = one month + one main category.** December and January only, same reason as above.
- Same columns and formulas as `gold_funnel`.
- A user who views products in 3 categories is counted in all 3 rows, so the category rows do **not** add up to the `gold_funnel` total.
- `unknown` (about 20% of events) is kept as its own row.
- Small categories (`medicine`, `stationery`, `country_yard`) are kept. Their rates still rest on hundreds of buyers. Check `viewers` before reading a rate.
- **Checks:** viewers ≥ carters ≥ buyers · all rates between 0 and 100

- ⚠️ **Category labels are not reliable.** About 46% of Apple events and 40% of Samsung events have a category the brand doesn't sell (e.g. Apple phones priced 443 to 1,597 labelled `construction.tools.light`). So findings and charts are **not** based on category. Brand and price are used instead.

## gold_funnel_brand
The same funnel as `gold_funnel`, split by **brand**.
- **One row = one month + one brand.** December and January only.
- Same columns and formulas. Rates use `nullif(..., 0)`, because some tiny brands have 0 carters.
- 5,325 brands. About half have fewer than 1,000 events, so their rates are not reliable. **Read rates only for big brands** (sort by `viewers`).
- `unknown` brand is kept as its own row.
- **Brand check:** only 1% of products (2,132 of 214,066) have more than one brand, so brand labels are reliable (unlike category).

In [0]:
%sql
use catalog dev;
use schema ecommerce

In [0]:
%sql
select * from events_silver limit 100

In [0]:
%sql
select count(*), event_type from events_silver
group by event_type


In [0]:
%sql
select 
count(distinct user_session) as orders
from events_silver
where user_session in (
    select user_session from events_silver
    where event_type = 'purchase'
)

In [0]:
%sql
select count(*) from events_silver
where price is null and event_type = 'purchase'

In [0]:
%sql
create or replace table dev.ecommerce.gold_orders as 
select 
concat(user_session, '_', user_id) as order_id, 
user_id as user_id,
min(event_time) as order_time,
count(*) as item_count,
sum(price) as order_value 
from dev.ecommerce.events_silver
where event_type = 'purchase' 
group by  
user_session, user_id 
 

In [0]:
%sql
select count(distinct order_id), count(*) from gold_orders

In [0]:
%sql
-- TEMP
select event_time, user_id, product_id, price
from dev.ecommerce.events_silver
where user_session = '93f1606e-2489-1208-967c-ad97c5f12330'
  and event_type = 'purchase'
order by event_time

In [0]:
%sql
select * from gold_orders limit 5

In [0]:
%sql
with x as (
select 
order_id, 
sum(item_count) as items from gold_orders
group by order_id 
order by items desc)

select count(order_id) from x 
where items >=10


In [0]:
%sql
-- TEMP
select item_count,
       count(*) as orders
from dev.ecommerce.gold_orders
group by item_count
order by item_count

In [0]:
%sql
-- TEMP
select sum(case when item_count >= 10 then order_value end) / sum(order_value) as big_order_revenue_share
from dev.ecommerce.gold_orders

In [0]:
%sql
select * from gold_orders limit 5

In [0]:
%sql
select 
date_format(order_time, 'yyyy-MM') as month,
count(order_id) as orders,
count(distinct user_id) as users 
from gold_orders
group by month 
order by month 

In [0]:
%sql
-- TEMP
select event_date,
       count_if(event_type = 'cart')     as carts,
       count_if(event_type = 'purchase') as purchases
from dev.ecommerce.events_silver
where event_date between '2019-11-01' and '2019-11-20'
group by event_date
order by event_date

In [0]:
%sql

create or replace table dev.ecommerce.gold_funnel as 
with x as (
    select date_format(event_date, 'yyyy-MM')                                    as month,
count(distinct case when event_type = 'view'     then user_id end)   as viewers,
count(distinct case when event_type = 'cart'     then user_id end)   as carters,
count(distinct case when event_type = 'purchase' then user_id end)   as buyers
from dev.ecommerce.events_silver
where event_date >= '2019-12-01'
group by month
)

select month,
viewers,
carters,
buyers,
viewers - carters                  as lost_at_cart,
carters - buyers                   as lost_at_purchase,
round(carters / viewers * 100, 1)  as cart_rate_pct,
round(buyers  / viewers * 100, 1)  as purchase_rate_pct,
round(buyers  / carters *100, 1)   as cart_to_purchase_pct
from x 
order by month

In [0]:
%sql
use catalog dev;
use schema ecommerce

In [0]:
%sql
select * from dev.ecommerce.gold_funnel 

In [0]:
%sql
select * from events_silver limit 5 

In [0]:
%sql
select 
category_code,
count(*) as events from events_silver
group by category_code 
order by events desc

In [0]:
%sql
-- TEMP: funnel by main category
SELECT
  split_part(category_code, '.', 1) AS main_category,
  count(DISTINCT CASE WHEN event_type = 'view'     THEN user_id END) AS viewers,
  count(DISTINCT CASE WHEN event_type = 'cart'     THEN user_id END) AS carters,
  count(DISTINCT CASE WHEN event_type = 'purchase'     THEN user_id END) AS buyers
FROM dev.ecommerce.events_silver
GROUP BY main_category
ORDER BY viewers DESC

In [0]:
%sql
select * from gold_funnel

In [0]:
%sql
CREATE OR REPLACE TABLE dev.ecommerce.gold_funnel_category AS
WITH x AS (
  SELECT
    date_format(event_date, 'yyyy-MM')     AS month,
    split_part(category_code, '.', 1)      AS main_category,
    count(DISTINCT CASE WHEN event_type = 'view'     THEN user_id END) AS viewers,
    count(DISTINCT CASE WHEN event_type = 'cart'     THEN user_id END) AS carters,
    count(DISTINCT CASE WHEN event_type = 'purchase' THEN user_id END) AS buyers
  FROM dev.ecommerce.events_silver
  WHERE event_date >= '2019-12-01'          -- Dec + Jan only (cart tracking)
  GROUP BY month, main_category
)
SELECT
  month,
  main_category,
  viewers,
  carters,
  buyers,
  viewers - carters                      AS lost_at_cart,
  carters - buyers                       AS lost_at_purchase,
  round(carters / viewers * 100, 1)      AS cart_rate_pct,
  round(buyers  / carters * 100, 1)      AS cart_to_purchase_pct,
  round(buyers  / viewers * 100, 1)      AS purchase_rate_pct
FROM x

In [0]:
%sql
select * from gold_funnel_category

In [0]:
%sql
select count(distinct brand) as brands from events_silver

In [0]:
%sql
select brand , round(count(*)/1e3,1) as nb_of_events_k from events_silver
group by brand
order by nb_of_events_k desc

In [0]:
%sql
CREATE OR REPLACE TABLE dev.ecommerce.gold_funnel_brand AS
WITH x AS (
  SELECT
    date_format(event_date, 'yyyy-MM') AS month,
    brand,
    count(DISTINCT CASE WHEN event_type = 'view'     THEN user_id END) AS viewers,
    count(DISTINCT CASE WHEN event_type = 'cart'     THEN user_id END) AS carters,
    count(DISTINCT CASE WHEN event_type = 'purchase' THEN user_id END) AS buyers
  FROM dev.ecommerce.events_silver
  WHERE event_date >= '2019-12-01'
  GROUP BY month, brand
)
SELECT
  month,
  brand,
  viewers,
  carters,
  buyers,
  viewers - carters                              AS lost_at_cart,
  carters - buyers                               AS lost_at_purchase,
  round(carters / nullif(viewers, 0) * 100, 1)   AS cart_rate_pct,
  round(buyers  / nullif(carters, 0) * 100, 1)   AS cart_to_purchase_pct,
  round(buyers  / nullif(viewers, 0) * 100, 1)   AS purchase_rate_pct
FROM x

In [0]:
%sql
-- TEMP: products with more than one brand
SELECT count(*) AS products_with_many_brands
FROM (
  SELECT product_id
  FROM events_silver
  WHERE brand <> 'unknown'
  GROUP BY product_id
  HAVING count(DISTINCT brand) > 1
)

In [0]:
%sql
-- TEMP
SELECT count(DISTINCT product_id) AS products
FROM events_silver
WHERE brand <> 'unknown'

In [0]:
%sql
-- TEMP
SELECT brand, split_part(category_code, '.', 1) AS main_category, count(*) AS events
FROM events_silver
WHERE brand IN ('samsung', 'apple', 'xiaomi', 'bosch', 'cordiant')
GROUP BY brand, main_category
ORDER BY brand, events DESC

In [0]:
%sql
-- TEMP
SELECT category_code, product_id, round(avg(price), 0) AS avg_price, count(*) AS events
FROM events_silver
WHERE brand = 'apple' AND category_code LIKE 'construction%'
GROUP BY category_code, product_id
ORDER BY events DESC
LIMIT 10

In [0]:
%sql
select brand, count(*) as events  from events_silver
group by brand
order by events desc

In [0]:
%sql
select * from gold_funnel_brand